#### ENRICH `ORDER_REVIEWS` FROM BRONZE TO SILVER

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERS_REVIEWS_ENRICHED'

In [0]:
order_reviews_df = spark.table('ORDERS_SALES.BRONZE.ORDERS_REVIEWS')
order_reviews_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

order_reviews_trans_df = (
    order_reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        col("file_path").alias("origin_file_path"),
        current_timestamp().alias("load_timestamp"),
    ).where("review_comment_title IS NOT NULL OR review_comment_message IS NOT NULL")
)
order_reviews_trans_df.limit(1).display()

In [0]:
order_reviews_trans_df.createOrReplaceTempView('order_reviews_trans_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDERS_REVIEWS_ENRICHED`

In [0]:
spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY review_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM order_reviews_trans_temp_vw
  )
  WHERE rn = 1
) src
ON (src.review_id = tgt.review_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_id = src.order_id,
    tgt.review_score = src.review_score,
    tgt.review_comment_title = src.review_comment_title,
    tgt.review_comment_message = src.review_comment_message
WHEN NOT MATCHED THEN
INSERT (review_id, order_id, review_score, review_comment_title, review_comment_message, origin_file_path,
load_timestamp)
VALUES (src.review_id, src.order_id, src.review_score, src.review_comment_title, src.review_comment_message, src.origin_file_path, current_timestamp)
''')

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDERS_REVIEWS_ENRICHED')